# 09 - E7c-A codebook plasticity

E7b cerró como resultado negativo útil: LayerNorm es seguro pero la memoria
latente vive aguas arriba de los LN, por lo que el regularizador no tenía
gradiente al codebook. E7c-A mueve la superficie de adaptación al
**`codebook_transform.weight`** de SimVQ (≈16k parámetros), la única matriz
aprendible que reescribe el codebook completo sin tocar BatchNorm.

**Muro estructural**: el straight-through estimator `q_st = z + (q - z).detach()`
bloquea el gradiente desde `zq` hacia el codebook. Por lo tanto la entropía de
logits (TENT/EATA puros) no llega al codebook. Las rutas vivas al
`codebook_transform.weight` son:
- `soft_assign = softmax(-distances/temp)` — entropía directa sobre `soft_assign`
  o el término KL del `latent_memory_loss`.
- `codebook_loss = MSE(q, z.detach())` — `q = one_hot @ emb` empuja el codebook
  hacia los `z` del dominio objetivo (adaptación real, no sólo ancla).

Variantes (TTA reset por celda `(corrupción, severidad)`):

- `source` — SimVQ E6 puro, sin adaptación.
- `bn_stats_no_update` — baseline literatura "BN Stats" (esperado colapso).
- `tent_codebook_softassign` — TENT minimizando entropía de `soft_assign`
  (camino con gradiente vivo al codebook).
- `tent_codebook_memreg` — entropía de logits + memoria latente; el término KL
  sobre `soft_assign` es lo único que muerde el codebook.
- `eata_codebook_srcfilter_memreg` — EATA filtrado por teacher + memoria.
- `codebook_loss_adapt` — minimiza el `codebook_loss = MSE(q, z.detach())`,
  empujando el codebook hacia el dominio objetivo (adaptación real).
- `codebook_loss_adapt_memreg` — idem + ancla `latent_memory_loss` (trade-off).
- `ttn_alpha_bn_010` / `ttn_alpha_bn_020` — E7c-D: α-mezcla de stats en el
  BatchNorm del projector (sin gradiente), para distinguir "ganancia por mezclar
  stats" de "ganancia por plasticidad del codebook".

**Variantes deliberadamente excluidas** (demostración del muro estructural en
los tests `test_tent_codebook_pure_is_structurally_inert`): `tent_codebook`
puro y `eata_codebook_srcfilter` puro — ambos producen una pérdida sin grafo
(`entropy.requires_grad=False`) cuando la única variable libre es el codebook.

**Diagnósticos extra contra teacher source congelado**: `z_drift`, `zq_drift`,
`assignment_churn`, `kl_assign_src`, `hard_usage_delta_vs_src`,
`dead_code_fraction_delta_vs_src`. El criterio de éxito primario es **mecánico**
(¿se mueve el codebook? ¿la mem-reg lo frena?), no de accuracy.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'dememte').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)

In [ ]:
import math

import numpy as np
import pandas as pd
import torch

from dememte.config import e6_config
from dememte.data import build_loaders, seed_everything
from dememte.evaluation import evaluate_dememte_suite, evaluate_dememte_tta_suite, signal_curve_rows
from dememte.io import ensure_dir, load_checkpoint, write_csv, write_json
from dememte.models import make_dememte_e6
from dememte.tta import (
    AlphaBNStatsAdapter,
    CodebookLossAdapter,
    MemoryTentAdapter,
    NoUpdateAdapter,
    SoftAssignTentAdapter,
    SourceFilterEATAAdapter,
    collect_tta_bn_params,
    collect_tta_codebook_params,
    configure_tta_codebook,
    configure_tta_model,
    make_tta_optimizer,
)

BASE_VARIANT = 'e6_simvq_linear'
METHODS = [
    'source',
    'bn_stats_no_update',
    'tent_codebook_softassign',
    'tent_codebook_memreg',
    'eata_codebook_srcfilter_memreg',
    'codebook_loss_adapt',
    'codebook_loss_adapt_memreg',
    'ttn_alpha_bn_090',
    'ttn_alpha_bn_095',
]

# Hiperparámetros TTA (idénticos a E7b para 1:1 comparabilidad).
TTA_LR = 2.5e-4
TTA_MOMENTUM = 0.9
EATA_D_MARGIN = 0.05
# Pesos del regularizador de memoria latente: (w_z, w_zq, w_assign).
MEM_WEIGHTS = (1.0, 1.0, 1.0)

OUT = ensure_dir(ROOT / 'notebooks' / '09_e7c_codebook' / 'out')
E6_OUT = ROOT / 'notebooks' / '06_e6_zq_alignment' / 'out'
CKPT = E6_OUT / BASE_VARIANT / 'best.pt'

cfg = e6_config(BASE_VARIANT)
for candidate in [ROOT / 'experiments' / 'data', ROOT / 'data', Path(cfg.data_dir).expanduser()]:
    candidate = candidate.resolve()
    if (candidate / 'flowers-102').exists() or candidate.name == 'flowers-102':
        cfg.data_dir = str(candidate)
        break

device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed_everything(cfg.seed)
print('device:', device)
print('checkpoint:', CKPT)
print('eata entropy margin:', 0.4 * math.log(cfg.num_classes))

## Data

In [ ]:
tr_loader, va_loader, te_loader, meta = build_loaders(
    data_dir=cfg.data_dir,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    val_ratio=cfg.val_ratio,
    split_seed=cfg.split_seed,
    protocol=cfg.benchmark_protocol,
)
print(meta)

## Evaluation helpers

In [ ]:
def write_markdown_table(rows, path):
    path = Path(path)
    ensure_dir(path.parent)
    if not rows:
        path.write_text('', encoding='utf-8')
        return
    df = pd.DataFrame(rows)
    # Manual pipe-table to avoid the optional 'tabulate' dependency.
    cols = list(df.columns)
    header = '| ' + ' | '.join(str(c) for c in cols) + ' |'
    sep = '| ' + ' | '.join('---' for _ in cols) + ' |'
    body = []
    for _, row in df.iterrows():
        cells = []
        for c in cols:
            v = row[c]
            if isinstance(v, float):
                cells.append(f'{v:.4f}')
            else:
                cells.append(str(v))
        body.append('| ' + ' | '.join(cells) + ' |')
    path.write_text('\n'.join([header, sep, *body]), encoding='utf-8')


def load_base_model():
    model = make_dememte_e6(cfg, device=device)
    load_checkpoint(model, CKPT, device=device, strict=True)
    return model


def make_adapter(method):
    # Baseline literatura "BN Stats": per-batch BN stats, sin paso de gradiente.
    if method == 'bn_stats_no_update':
        model = configure_tta_model(load_base_model())
        params, _ = collect_tta_bn_params(model)
        opt = make_tta_optimizer(params, lr=TTA_LR, momentum=TTA_MOMENTUM)
        return NoUpdateAdapter(model, opt)

    # E7c-D: TTN/alpha-BN sobre projector.net.1, sin gradiente.
    # alpha es el peso del source running stats: alpha=1.0 es no-op, alpha=0.0
    # reproduce bn_stats_no_update (colapso). Usamos alpha alto (0.90/0.95)
    # para preservar el modelo e inyectar sólo una pequeña correccion batch.
    if method == 'ttn_alpha_bn_090':
        return AlphaBNStatsAdapter(load_base_model(), alpha=0.90)
    if method == 'ttn_alpha_bn_095':
        return AlphaBNStatsAdapter(load_base_model(), alpha=0.95)

    # E7c-A: superficie codebook_transform.weight (SimVQ).
    model = configure_tta_codebook(load_base_model())
    params, _ = collect_tta_codebook_params(model)
    opt = make_tta_optimizer(params, lr=TTA_LR, momentum=TTA_MOMENTUM)

    if method == 'tent_codebook_softassign':
        return SoftAssignTentAdapter(model, opt, steps=1, episodic=False)
    if method == 'tent_codebook_memreg':
        return MemoryTentAdapter(
            model, opt, source_model=load_base_model(), steps=1, episodic=False,
            w_z=MEM_WEIGHTS[0], w_zq=MEM_WEIGHTS[1], w_assign=MEM_WEIGHTS[2],
        )
    if method == 'eata_codebook_srcfilter_memreg':
        return SourceFilterEATAAdapter(
            model, opt, num_classes=cfg.num_classes, source_model=load_base_model(),
            steps=1, episodic=False, d_margin=EATA_D_MARGIN, memory_weights=MEM_WEIGHTS,
        )
    if method == 'codebook_loss_adapt':
        return CodebookLossAdapter(model, opt, steps=1, episodic=False)
    if method == 'codebook_loss_adapt_memreg':
        return CodebookLossAdapter(
            model, opt, source_model=load_base_model(), steps=1, episodic=False,
            memory_weights=MEM_WEIGHTS,
        )
    raise ValueError(method)


def summarize_metrics(method, metrics):
    summary = {k: v for k, v in metrics.items() if isinstance(v, (int, float, bool, str, np.floating))}
    summary.update({
        'variant': method,
        'label': method,
        'base_variant': BASE_VARIANT,
        'base_checkpoint': str(CKPT),
        'protocol': meta['protocol'],
        'split_seed': meta['split_seed'],
        'quantizer_type': cfg.quantizer_type,
    })
    return summary

## Run E7c

In [ ]:
if not CKPT.exists():
    raise FileNotFoundError(f'Missing E6 SimVQ checkpoint: {CKPT}')

# Teacher source congelado, instanciado una sola vez (no necesita reset por celda).
teacher_model = load_base_model().eval()
teacher_model.requires_grad_(False)

all_summaries = []
all_curves = []

for method in METHODS:
    print(f'=== {method} ===')
    if method == 'source':
        model = load_base_model()
        metrics = evaluate_dememte_suite(model, te_loader, device=device)
    else:
        metrics = evaluate_dememte_tta_suite(
            lambda method=method: make_adapter(method),
            te_loader,
            device=device,
            tta_method=method,
            tta_base_variant=BASE_VARIANT,
            teacher_model=teacher_model,
        )

    clean_record = metrics.pop('clean_record')
    corrupt_records = metrics.pop('corruption_records')
    curve_rows = signal_curve_rows(method, method, clean_record, corrupt_records)
    summary = summarize_metrics(method, metrics)
    all_summaries.append(summary)
    all_curves.extend(curve_rows)

    method_dir = ensure_dir(OUT / method)
    write_json(summary, method_dir / 'metrics.json')
    write_csv(curve_rows, method_dir / 'signal_curves.csv')
    report_keys = [
        'clean_acc', 'corrupt_acc_avg', 'ece_corrupt_avg', 'nll_corrupt_avg',
        'hard_usage_corrupt_avg', 'dead_code_fraction_corrupt_avg',
        'zq_drift_corrupt_avg', 'assignment_churn_corrupt_avg',
    ]
    print({k: round(float(summary[k]), 4) for k in report_keys if k in summary})

write_csv(all_summaries, OUT / 'e7c_results.csv')
write_csv(all_curves, OUT / 'e7c_curves.csv')

ranked = sorted(all_summaries, key=lambda r: r.get('corrupt_acc_avg', 0.0), reverse=True)
write_markdown_table(ranked, OUT / 'e7c_summary.md')
pd.DataFrame(ranked)

## Read

**Hito mecánico (criterio principal, no accuracy).** El orden esperado de
los drifts contra teacher source en corrupciones es:

1. `source` y `ttn_alpha_bn_*` — `zq_drift_corrupt_avg = 0` (no tocan el
   codebook ni los embeddings).
2. `bn_stats_no_update` — drift indefinido (modelo colapsa).
3. `tent_codebook_softassign` — `zq_drift > 0`, `assignment_churn > 0`:
   el codebook es **mojable** desde TTA cuando la pérdida toca `soft_assign`.
4. `tent_codebook_memreg` y `eata_codebook_srcfilter_memreg` — drift
   **atenuado** respecto a `tent_codebook_softassign`: el regularizador de
   memoria latente muerde via KL en `soft_assign` (primer dato real sobre Q5).
5. `codebook_loss_adapt` — drift positivo (probablemente más alto que
   `softassign`): el codebook se mueve **hacia los `z` del target**
   (adaptación real, no sólo ancla).
6. `codebook_loss_adapt_memreg` — drift positivo pero menor que sin memreg:
   trade-off explícito plasticidad ↔ preservación.

**Comparativa con `ttn_alpha_bn_*`**: si `ttn_alpha_bn_010/020` ya capturan
la mayoría de la ganancia (en `corrupt_acc_avg`), entonces buena parte de
lo que las variantes de codebook logran es atribuible a "mezclar stats",
no a plasticidad del codebook. Si las variantes `_memreg` superan a TTN sin
moverse mucho del codebook (drift bajo), eso es la señal positiva de Q5:
la memoria preservada **ayuda activamente**, no es decorativa.

**Notas sobre el muro estructural.** Las variantes `tent_codebook` puro y
`eata_codebook_srcfilter` (sin memreg) NO están en el grid. Cuando la única
variable libre es `codebook_transform.weight`, la entropía de logits queda
completamente desconectada del grafo (`requires_grad=False`) porque el
straight-through `q_st = z + (q - z).detach()` corta la única ruta a
embedding. Esto se demuestra en
`tests/test_vqsa.py::test_tent_codebook_pure_is_structurally_inert`.